```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b done;
    class A5a current;
    class A5b,A6a,A6b,A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Notebook 05a — Preprocessing and Annotation (run at home)

## **Purpose:**  
 This notebook loads the cleaned corpus, applies spaCy’s linguistic annotation pipeline (tokenization, lemmatization, POS tagging, dependency parsing), and saves the annotated documents in a reusable format. It will be run as a homework before session 6 because the annotation process can take 1‑2 hours on a laptop.
  
**The notebook outputs are:**  
- A folder (`data/processed/nb05-corpus-split/`) containing several `.spacy` files, each holding a batch of annotated `Doc` objects.  
- Two CSV summary tables with token and sentence counts per document.  
 
**After running this notebook**, you can shut down the kernel and proceed to Notebook 05b.

# Automated text annotation and conceptual relations

## Overview

This notebook marks a pivotal transition in the pipeline: we move from **lexical exploration** to **structured linguistic representation**. We will use an NLP specialised library called spaCy [spaCy](https://spacy.io/) to build an automated annotation layer that transforms raw text into richly labelled linguistic objects — tokens, lemmas, part-of-speech tags, and dependency parses. We will use this structured outputs to extract **interpretable conceptual relations** from the corpus.

The outputs produced here are designed to be **reusable artifacts**: serialised annotations and relation tables that downstream notebooks can load directly, without re-running the NLP pipeline.

---

## Learning Objectives

By the end of this notebook you will be able to:

1. **Load and align** the processed corpus and metadata produced in previous notebooks.
2. **Construct a reusable spaCy annotation pipeline** and apply it at scale to the full corpus.
3. **Perform core NLP tasks** — tokenization, sentence segmentation, lemmatization, POS tagging, and dependency parsing — and inspect their outputs critically.
4. **Extract four families of conceptual relations** that surface meaningful semantic structure:

   | Relation type | Example | What it captures |
   |---|---|---|
   | **Adjective–noun** | *computational linguistics* | Properties attributed to concepts |
   | **Subject–verb–object triples** | *researchers → develop → models* | Actor–action–patient structure |
   | **Noun compounds** | *language model* | Multi-word concept formation |
   | **Concept co-occurrence windows** | *syntax* ↔ *semantics* (within *n* tokens) | Topical proximity between concepts |

5. **Serialize all outputs** (annotated docs, relation tables) for reuse in later notebooks.

---

## Methodological Note

> **Automated annotation is probabilistic and model-dependent.**
> spaCy's statistical models produce high-quality approximations, but they are not infallible — accuracy varies by genre, domain, and sentence complexity. Treat the outputs as *useful structured estimates* rather than ground-truth linguistic analyses, and keep this caveat in mind when interpreting downstream results.


## spaCy key terminology

> Before we begin coding, make sure you are comfortable with the following spaCy concepts. If any of them feel unfamiliar, work through the [spaCy 101](https://spacy.io/usage/spacy-101) guide.

### Core Objects

| Object | What it is | Typical access |
|---|---|---|
| **`nlp`** | A `Language` pipeline object. When you call `nlp(text)` it tokenizes the text and runs every pipeline component (tagger, parser, NER…) in order, returning a `Doc`. | `nlp = spacy.load("en_core_web_sm")` |
| **`Doc`** | An ordered sequence of `Token` objects — the container for all annotations produced by the pipeline. A `Doc` is created by calling `nlp(text)` and is the primary unit you will work with. | `doc = nlp("Chomsky proposed transformational grammar.")` |
| **`Token`** | A single token (roughly, a word or punctuation mark). Each token carries attributes such as `.text`, `.lemma_`, `.pos_`, `.dep_`, and `.head`. | `for token in doc:` |
| **`Span`** | A slice of a `Doc` — one or more contiguous tokens. Named entities (`doc.ents`) and sentences (`doc.sents`) are returned as `Span` objects. | `doc[0:3]` or `ent in doc.ents` |
| **`Vocab`** | A shared vocabulary store that maps strings to integer hashes for memory efficiency. You rarely interact with it directly, but it explains why spaCy attributes come in two forms (e.g., `.pos` → integer hash, `.pos_` → human-readable string). | `doc.vocab` |

### Linguistic Attributes (per Token)

| Attribute | Description | Example value |
|---|---|---|
| `.text` | The original surface form of the token. | `"proposed"` |
| `.lemma_` | The base/dictionary form (lemma). | `"propose"` |
| `.pos_` | Coarse-grained part-of-speech tag ([Universal POS tags](https://universaldependencies.org/u/pos/)). | `"VERB"` |
| `.tag_` | Fine-grained, language-specific POS tag. | `"VBD"` |
| `.dep_` | Syntactic dependency relation to the token's `.head`. | `"nsubj"`, `"dobj"` |
| `.head` | The syntactic governor (parent) of the token in the dependency tree. | another `Token` |
| `.is_stop` | Whether the token belongs to spaCy's built-in stop-word list. | `True` / `False` |
| `.is_alpha` | Whether the token consists entirely of alphabetic characters. | `True` / `False` |

### Pipeline Components

A spaCy pipeline is a sequence of components applied to each `Doc` in order. The default English pipelines typically include:

- **Tokenizer** — splits raw text into tokens (always runs first; not a named pipeline component).
- **Tagger** (`tagger`) — assigns `.pos_` and `.tag_` attributes.
- **Dependency Parser** (`parser`) — builds the syntactic dependency tree (`.dep_`, `.head`) and identifies sentence boundaries (`.sents`).
- **Lemmatizer** (`lemmatizer`) — assigns `.lemma_` (requires the tagger or morphologizer to run first).
- **Named Entity Recognizer** (`ner`) — identifies named entities and stores them in `doc.ents` *(we will explore this in detail in the next notebook)*.

You can inspect the active components of any pipeline with:

```python
print(nlp.pipe_names)
# e.g. ['tok2vec', 'tagger', 'parser', 'lemmatizer', 'ner']
```

### Trained Pipelines (Models)

spaCy ships statistical pipelines of different sizes.

| Pipeline | Size | Includes |
|---|---|---|
| `en_core_web_sm` | ~12 MB | Tagger, parser, lemmatizer, NER (no word vectors) |
| `en_core_web_md` | ~40 MB | Same + 300-d word vectors (685 k keys) |
| `en_core_web_lg` | ~560 MB | Same + 300-d word vectors (685 k keys, more coverage) |

> **Which one should I use?** For annotation tasks in this notebook, `en_core_web_sm` is sufficient and fast. For word-vector similarity features, switch to `md` or `lg`.


# Setup

In [ ]:
!pip install spacy

In [ ]:
# -----------------------------
# Import
# -----------------------------
from __future__ import annotations

from pathlib import Path
import re
from collections import Counter

import numpy as np
import pandas as pd

import spacy
from spacy.tokens import DocBin

from tqdm.auto import tqdm

# -----------------------------
# Paths
# -----------------------------
PROJECT_ROOT = Path('.')

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
TEXTS_DIR = PROCESSED_DIR / 'cleaned'          

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
TABLES_DIR = ANALYSIS_DIR / 'tables'

CACHE_DIR = PROJECT_ROOT / 'cache'

# Path to the document index created in Notebook 03
DOC_INDEX = TABLES_DIR / 'nb03-doc_index.csv'

print('TEXTS_DIR:', TEXTS_DIR)
print('DOC_INDEX:', DOC_INDEX)

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------

def read_clean_text(texts_dir: Path, pg_id: int) -> str:
    filename = f"pg{pg_id}.txt"
    return (texts_dir / filename).read_text(encoding="utf-8", errors="replace")


def snippet(text: str, n: int = 180) -> str:
    """Return a short snippet of a text for preview purposes."""
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return text[:n] + ('…' if len(text) > n else '')


def chunk_text(text: str, max_chars: int) -> list[str]:
    """
    Split a long text into chunks of approximately `max_chars` characters.
    Breaks at newlines or spaces to avoid cutting words.
    """
    if len(text) <= max_chars:
        return [text]
    
    chunks = []
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = min(start + max_chars, text_len)
        
        if end < text_len:
            # Try to break at a newline, else at a space
            break_pos = text.rfind('\n', start, end)
            if break_pos == -1:
                break_pos = text.rfind(' ', start, end)
            if break_pos == -1 or break_pos <= start:
                break_pos = end
            chunks.append(text[start:break_pos])
            start = break_pos
            # Skip whitespace at the beginning of the next chunk
            while start < text_len and text[start].isspace():
                start += 1
        else:
            chunks.append(text[start:end])
            start = end
    
    return chunks

In [ ]:
# Load the corpus metadata
df = pd.read_csv(DOC_INDEX)
# Convert years from float to int
df["publication_year"] = pd.to_numeric(df["publication_year"], errors="coerce").astype("Int64")

# Uncomment below to limit the number of documents for testing
# MAX_DOCS = 10
# if MAX_DOCS is not None:
#     df = df.head(MAX_DOCS).copy()

print(f"\n{len(df)} documents' metadata loaded.\n")
display(df.head(3))

# Add the text column by reading each file
df['text'] = df['pg_id'].apply(lambda fn: read_clean_text(TEXTS_DIR, fn))

print('\nLoaded cleaned texts.')
print('\nExample snippet:')
print(snippet(df.iloc[0]['text']))

### Load the spaCy Pipeline

 We use the small English model `en_core_web_sm`. For this notebook we keep the full pipeline
 (tagger, parser, NER) because we need POS and dependency relations.
 However, we will later split the processing into chunks to handle very long texts.

In [ ]:
# Downolad spacy model
!python -m spacy download en_core_web_sm

In [ ]:
# Create spacy nlp object
nlp = spacy.load("en_core_web_sm")
nlp.max_length = 1500000   # safety net, though we chunk the texts

print('\nPipeline components:', nlp.pipe_names)

### Process the Corpus in Chunks
 
To avoid memory errors on long texts (>1M characters), we split each document
into smaller chunks of ~200,000 characters. We then process each chunk with spaCy,
store the resulting `Doc` objects in a `DocBin`, and keep metadata about each chunk.

In [ ]:
# Configuration for chunking and processing
CHUNK_SIZE = 200000          # characters per chunk – safe for laptops
BATCH_SIZE = 10              # chunks per batch for spaCy's pipe
N_PROCESS = 1                # keep at 1 to avoid memory overhead on laptops

In [ ]:
# Prepare chunk records: list of (original_df_index, chunk_index, chunk_text)
print("Splitting texts into chunks...")
chunk_records = []
for df_idx, row in df.iterrows():
    raw_text = str(row['text'])
    chunks = chunk_text(raw_text, CHUNK_SIZE)
    for chunk_i, chunk_txt in enumerate(chunks):
        chunk_records.append((df_idx, chunk_i, chunk_txt))

print(f"Total chunks generated: {len(chunk_records)}")

# Process each chunk and build the DocBin
meta_rows = []                          # one row per chunk
docbin = DocBin(store_user_data=True)   # store metadata for each Doc

for i, (orig_df_idx, chunk_i, chunk_text) in enumerate(tqdm(chunk_records, desc="Processing chunks")):
    # Process the chunk
    doc = nlp(chunk_text)
    
    # Retrieve original metadata
    orig_row = df.iloc[orig_df_idx]
    
    # --- Store metadata on the Doc ---
    doc.user_data['pg_id'] = int(orig_row['pg_id']) if pd.notna(orig_row['pg_id']) else None
    doc.user_data['title'] = orig_row['title']
    doc.user_data['publication_year'] = int(orig_row['publication_year']) if pd.notna(orig_row['publication_year']) else None
    doc.user_data['time_bin'] = orig_row['time_bin']
    doc.user_data['chunk_index'] = chunk_i  # Track each chunk
    
    # Add to the DocBin
    docbin.add(doc)
    
    # Build metadata row for this chunk
    meta_rows.append({
        'chunk_id': i,
        'original_doc_id': orig_df_idx,
        'chunk_index': chunk_i,
        'pg_id': orig_row.get('pg_id', pd.NA),
        'filename': f"pg{orig_row.get('pg_id', pd.NA)}.txt",
        'title': orig_row.get('title', pd.NA),
        'publication_year': orig_row.get('publication_year', pd.NA),
        'time_bin': orig_row.get('time_bin', pd.NA) if 'time_bin' in df.columns else pd.NA,
        'n_tokens': len(doc),
        'n_sentences': sum(1 for _ in doc.sents),
    })

# Convert metadata to DataFrame
meta_df = pd.DataFrame(meta_rows)
print(f"\nProcessed {len(meta_df)} total chunks across {df.shape[0]} original documents.")

# Aggregate metadata per original document
aggregated_meta = meta_df.groupby('original_doc_id').agg({
    'pg_id': 'first',
    'filename': 'first',
    'title': 'first',
    'publication_year': 'first',
    'time_bin': 'first',
    'n_tokens': 'sum',
    'n_sentences': 'sum',
    'chunk_index': 'count'          # number of chunks for this doc
}).rename(columns={'chunk_index': 'n_chunks'}).reset_index(drop=False)

print("\n--- Aggregated Metadata (Per Original Document) ---")
display(aggregated_meta.head())

### Save the metadata tables


In [ ]:
# The metadata CSVs are small and can be loaded later for reference.

META_PATH = TABLES_DIR / 'nb05-doc_annotation_summary.csv'
AGG_META_PATH = TABLES_DIR / 'nb05-doc_agg_annotation_summary.csv'

meta_df.to_csv(META_PATH, index=False)
aggregated_meta.to_csv(AGG_META_PATH, index=False)

print('\nSaved metadata table:', META_PATH)
print('Saved aggregated metadata table:', AGG_META_PATH)

### Split the DocBin into Smaller Files
 
The single spacy `DocBin` may be too large to save as one file (it can exceed spaCy's limit).
We split it into batches of 100 documents (chunks) and save each as a separate `.spacy` file.
This also makes loading faster and more memory-efficient.

In [ ]:
DOCS_PER_BIN = 100
SPLIT_DIR = PROCESSED_DIR / 'nb05-corpus-split'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("\nSplitting the DocBin into smaller files...")
batch_num = 0
doc_count_in_batch = 0
total_docs = 0
batch_docbin = DocBin(store_user_data=True)

# Iterate over the documents already in the DocBin (still in memory)
for doc in tqdm(docbin.get_docs(nlp.vocab), total=len(docbin), desc="Splitting"):
    batch_docbin.add(doc)
    doc_count_in_batch += 1
    total_docs += 1
    if doc_count_in_batch >= DOCS_PER_BIN:
        batch_path = SPLIT_DIR / f'batch_{batch_num:03d}.spacy'
        batch_docbin.to_disk(batch_path)
        print(f"  -> Saved {batch_path} ({doc_count_in_batch} docs)")
        batch_docbin = DocBin(store_user_data=True)
        batch_num += 1
        doc_count_in_batch = 0

# Save the final partial batch
if doc_count_in_batch > 0:
    batch_path = SPLIT_DIR / f'batch_{batch_num:03d}.spacy'
    batch_docbin.to_disk(batch_path)
    print(f"  -> Saved {batch_path} ({doc_count_in_batch} docs)")
    batch_num += 1

print(f"\nSuccessfully split {total_docs} documents into {batch_num} files.")
print(f"Files are located in: {SPLIT_DIR}")

In [ ]:
# Free Memory
# Now that the data is safely saved to disk, we can delete the large `docbin`
# to free up RAM for other tasks. This is not required but recommended.
del docbin
print("Deleted the large DocBin from memory.")

## Next Steps

You have successfully annotated the entire corpus and saved the results.
 - The annotated documents are in: `data/processed/nb05-corpus-split/`
 - The metadata summaries are in: `analysis/tables/nb05-*.csv`
 
You can now close this notebook and move to **Notebook 05b**,
which loads these saved files and performs all the conceptual relation analyses.

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A5a highlight;
```